# 01 · Prognosis is not treatment effect

**Left off (2026-10-11).** Open questions from Shaun:
- What exactly does "prognosis is not treatment effect" mean?
- Why do we need a toy example? Why not use the existing data instead of made-up data?

**Synthetic data.** Parts 1 and 2 use made-up patients that this notebook creates. Part 3 uses the Kaggle CIBMTR data, which is also synthetic.

## Key to abbreviations

| Term | Full name | Meaning here |
|---|---|---|
| HCT | Hematopoietic cell transplantation | A transplant of blood-forming stem cells, for example a bone marrow transplant. |
| CIBMTR | Center for International Blood and Marrow Transplant Research | The research group that hosts the Kaggle competition. |
| EFS | Event-free survival | The outcome. An "event" is relapse, graft failure or death. |
| Conditioning | — | Chemotherapy and/or radiation before the transplant. It clears the old marrow. |
| MAC | Myeloablative conditioning | High-dose conditioning. Better disease control, but more toxic. |
| RIC | Reduced-intensity conditioning | Lower-dose conditioning. Less toxic, so doctors give it to older and frailer patients. |
| DCA | Decision curve analysis | A method that shows if a model gives better decisions than "act for all" or "act for none". See Part 2. |
| p_t | Threshold probability | The risk above which you act. See Part 2. |
| GAN | Generative adversarial network | A type of model that makes synthetic data. SurvivalGAN is the GAN that made the Kaggle data. |
| Cox | Cox proportional hazards model | A standard regression model for survival data. A name, not an acronym. |

## The idea in one example

People who go to the hospital die more often than people who stay at home. This does not mean that hospitals kill people. Sick people go to the hospital. So the comparison mostly shows who was sick. It does not show what the hospital does.

This notebook makes the same point about transplant treatments. Change "hospital" to "MAC or RIC". Change "sick" to "frail" or "severe disease".

## Two different questions

| Question | Name | Example |
|---|---|---|
| "How likely is a bad outcome for this patient?" | **Prognosis** | "This patient has a 40% chance of an EFS event within 12 months." |
| "Will this patient do better with treatment A or with treatment B?" | **Treatment effect** | "With RIC instead of MAC, this patient's chance goes from 40% to 30%." |

A **risk model** answers the first question. It takes facts about a patient and gives the chance of a bad outcome. Another name for it is a **prognostic model**. Our competition models (Cox, gradient-boosted survival) are risk models.

## The mistake this notebook shows

Our risk models use the treatment as one of their inputs, for example `conditioning_intensity` (MAC or RIC). So you can do this:

1. Give the model a patient's facts with MAC. Read the predicted risk.
2. Give the model the same facts, but with RIC. Read the predicted risk again.
3. Compare the two risks. Choose the treatment with the lower risk.

This looks like an answer to the treatment question. **It is not a reliable answer.** The model did not learn what RIC does. It learned how the patients who got RIC did. These are different things, because doctors do not choose the treatment at random.

## Why the doctor's choice matters

Doctors choose a treatment for a reason. For example, they give RIC to older and frailer patients. These patients do badly with any treatment. So the RIC group does badly, even if RIC is the better treatment for them. This is the hospital example again.

The name for this problem is **confounding**. A **confounder** is a fact about the patient that changes both:
- which treatment the patient gets, and
- how the patient does.

When the doctor's reason for the treatment is the confounder, the name is **confounding by indication**.

A model can correct for a confounder only if the confounder is in the data. Age is usually recorded, so the model can correct for age. But doctors also see things that nobody records. The model cannot correct for those.

A risk model can still predict well. To predict, it only needs to know *that* a patient's risk is high. It does not need to know *why*. A treatment choice needs to know why.

## Plan

| Part | Data | What it shows |
|---|---|---|
| 1 | Made-up patients where we know the correct answer | The risk model gives wrong treatment advice. |
| 2 | The same made-up patients | The same risk model is still useful for a different decision. |
| 3 | Kaggle CIBMTR data | The same warning signs are in the competition data. |

In [ ]:
from hct_equity import teaching
from hct_equity.config import load_config
from hct_equity.data import load_train
import dcurves

project = load_config()
settings = project.teaching_simulation
print(project.synthetic_note)

## Part 1 · Made-up patients with a known answer

### Why make up patients?

With real patients, each patient gets only one treatment. We never see what would have happened with the other treatment. So we can never check if a method gives correct treatment advice.

With made-up patients, we write the rules ourselves. So we know each patient's risk with MAC **and** with RIC. This gives us an **answer key**. We can test each method against the answer key. The rules are in `config/project.json`, under `teaching_simulation`.

### What each patient has

| Column | Meaning | Can a real analysis see it? |
|---|---|---|
| `frail` | 1 = frail patient, 0 = fit patient. Frail patients do worse. | Yes |
| `severe` | 1 = aggressive disease. These patients do worse. The doctor sees it. | **No.** Nobody recorded it. |
| `ric` | The treatment that the doctor chose: 1 = RIC, 0 = MAC. | Yes |
| `risk_mac` | The patient's true chance of an event with MAC. | No. Only the answer key has it. |
| `risk_ric` | The patient's true chance of an event with RIC. | No. Only the answer key has it. |
| `event` | 1 = the patient had an event with the treatment they got. | Yes |

### The rules we chose

How the doctor chooses:
- Frail patients usually get RIC.
- Patients with severe disease usually get MAC.

What the treatment does (the answer key):
- For **frail** patients, RIC is better. MAC is too toxic for them.
- For **fit** patients, MAC is better. It controls the disease better.

So the best treatment is different for different patients. A good method must find this.

The table below shows averages for each group of patients. In the `ric` column, the average is the share of the group that got RIC. For example, 0.8 means that 80% of the group got RIC.

In [ ]:
cohort = teaching.simulate_cohort(settings)
cohort.groupby(["frail", "severe"])[["ric", "risk_mac", "risk_ric"]].mean().round(2)

Frail patients usually got RIC. Patients with severe disease rarely got RIC. This matches the rules.

### Method 1: the naive comparison

"Naive" means simple, with no correction. Compare the event rate of the patients who got RIC with the event rate of the patients who got MAC. This is the hospital-versus-home comparison.

In [ ]:
cohort.groupby("ric")["event"].mean().rename("event_rate").round(3)

RIC looks **worse**: the RIC group has more events. But most RIC patients are frail, and frail patients do badly with any treatment. This comparison shows *who got RIC*. It does not show *what RIC does*.

### Method 2: the risk model

Now we fit a risk model. The model is a **logistic regression**. This is a standard formula that turns patient facts into a chance between 0 and 1:

> risk score = a + b × frail + c × ric + d × (frail × ric)

**Fitting** means that the computer picks the values of a, b, c and d that best match the events in the data. Each of these values is a **coefficient**. The last term, `frail × ric`, lets the effect of RIC be different for frail and fit patients. The model sees `frail` and `ric`. It does not see `severe`.

Then we use the model for treatment advice, in three steps:
1. Copy all patients, and set every copy to MAC. The model predicts each patient's risk with MAC.
2. Copy all patients again, and set every copy to RIC. The model predicts each patient's risk with RIC.
3. Subtract: (risk with RIC) − (risk with MAC). A negative result means that the model says "RIC helps".

The code below does these steps. Then it shows the average result for fit patients and for frail patients.

In [ ]:
model = teaching.fit_prognostic(cohort, teaching.OBSERVED_FEATURES)
teaching.effect_by_frailty(cohort, model, teaching.OBSERVED_FEATURES).round(3)

### How to read the table

Each number is (risk with RIC) − (risk with MAC). A **negative** number means that RIC lowers the risk. A **positive** number means that RIC raises the risk.

| Column | Where it comes from | Exists for real patients? |
|---|---|---|
| `true_effect` | The answer key: `risk_ric` − `risk_mac` for each patient. | No |
| `model_effect` | Method 2: the risk model, with the treatment input switched. | Yes |
| `naive_effect` | Method 1: event rate with RIC − event rate with MAC. Done separately for fit and frail patients. | Yes |

Row `frail = 0` is the fit patients. Row `frail = 1` is the frail patients.

**For fit patients, the model gives the wrong answer.** `true_effect` is positive: RIC harms fit patients. `model_effect` is negative: the model says that RIC helps them.

For frail patients, the model has the correct direction. But it says that RIC helps more than it does.

### Why `model_effect` and `naive_effect` are the same

The formula has four coefficients. There are four groups of patients: fit with MAC, fit with RIC, frail with MAC, frail with RIC. So the model can match the event rate of each group exactly. Its predictions are only the group averages. So the model gives the same answer as the naive comparison. Here, the model adds nothing to the simple comparison.

### Why the model is wrong

The cause is `severe`. The table below counts the fit patients by treatment (`ric`) and by `severe`.

In [ ]:
cohort[cohort["frail"] == 0].groupby(["ric", "severe"]).size().unstack()

Almost all fit patients with severe disease got MAC (row `ric = 0`, column `severe = 1`). Severe disease gives a high risk with any treatment. So the MAC group has most of the sickest patients, and it has more events.

The model cannot see `severe`. It sees that MAC patients have more events, and it blames MAC. In truth, severe disease causes those events.

This error is **bias**. Bias is an error that always goes in the same direction. Random error from a small sample becomes smaller with more data. Bias does not. We have many patients here, so the error is not bad luck. The hidden `severe` causes it.

### What if we choose treatments with the model?

A **policy** is a rule that chooses a treatment for each patient. The table below compares five policies:

| Policy | Rule |
|---|---|
| Current practice | Each patient gets the treatment that the doctor chose. |
| Everyone MAC | All patients get MAC. |
| Everyone RIC | All patients get RIC. |
| Choose with prognostic model | Each patient gets the treatment with the lower risk from the model. |
| Choose with true risks | Each patient gets the treatment with the lower risk from the answer key. This is the best possible policy. |

`event_risk` is the average true chance of an event if all patients follow that policy. Lower is better. `pct_ric` is the percent of patients who get RIC. We can calculate `event_risk` only because we have the answer key.

In [ ]:
teaching.compare_policies(cohort, model, teaching.OBSERVED_FEATURES).round(3)

The model's policy gives **RIC to every patient**. This has a higher event risk than current practice. The doctors see `severe`, so they already choose better than the model.

### Check: give the model the hidden column

What if `severe` were in the data? The next code cell fits the model again, with `severe` as an input. Then it shows the same two tables again.

In [ ]:
full_model = teaching.fit_prognostic(cohort, teaching.ALL_FEATURES)
display(teaching.effect_by_frailty(cohort, full_model, teaching.ALL_FEATURES).round(3))
teaching.compare_policies(cohort, full_model, teaching.ALL_FEATURES).round(3)

Now `model_effect` is close to `true_effect`, and the model's policy matches the best policy. So the hidden `severe` caused the error.

**Lesson:** a model gives correct treatment advice only if the data has **every** fact that changes both the treatment choice and the outcome. The name for this condition is **no unmeasured confounding**. Real data cannot show if it is true. It is an assumption, and you cannot test it with the data.

## Part 2 · What DCA can and cannot answer

### A different decision: act or do not act

Some decisions have only two options: do something, or do nothing. Example: **extra check-ups after the transplant**. Check-ups are cheap and safe. So it is acceptable to give some check-ups to patients who do not need them.

For this decision, the model only needs to put patients in order of risk. It does not need to know *why* a risk is high. So the biased model from Part 1 can still help here.

### What DCA does

**DCA** (decision curve analysis) checks if a model gives better act-or-not decisions. It compares three choices:
- **Treat all**: act for every patient.
- **Treat none**: act for no patient.
- **Use the model**: act only for patients with a predicted risk above a cut-off.

The cut-off is the **threshold probability**, p_t. It shows how you balance harm and benefit:

> p_t / (1 − p_t) = harm of an unnecessary action ÷ benefit of a necessary action

Example: p_t = 0.1 gives 0.1 / 0.9 = 1/9. So you accept up to nine unnecessary check-ups to give one necessary check-up. Cheap, safe actions have low thresholds.

For each threshold, DCA calculates the **net benefit**. Net benefit counts the necessary actions, and subtracts the unnecessary actions weighted by the threshold. Higher is better.

In [ ]:
thresholds = project.teaching_dca_thresholds.values()
dca_result = teaching.surveillance_dca(cohort, model, teaching.OBSERVED_FEATURES, thresholds)
dcurves.plot_graphs(plot_df=dca_result, graph_type="net_benefit", y_limits=(-0.05, 0.7))

### How to read the plot

The x-axis is the threshold. The y-axis is the net benefit. Higher is better.
- **Treat none** is 0 at all thresholds.
- **Treat all** is high at low thresholds, because about 59% of the made-up patients have an event. When an action is cheap and most patients have an event, acting for all patients is a strong default.
- At low thresholds, every patient's predicted risk is above the threshold. So the model acts for every patient, and its line **equals** treat all. The model adds value only where it acts for some patients and not for others.
- At higher thresholds, the model line is above both other lines. In that range, the model helps with the check-up decision.

The same model gave bad treatment advice in Part 1. **One model can be useful for one decision and harmful for another.**

The CIBMTR data is similar: about 52% of patients have an event within 12 months. So treat all is strong at low thresholds, and our models must do better than it.

### Which tool fits which decision

| Decision | What the model must get right | Can a risk model do it? | How to check |
|---|---|---|---|
| Act or do not act (check-ups, referral, screening for a trial) | Which patients have a high risk | Yes | DCA |
| Choose between treatments (MAC or RIC, donor type) | How each patient's risk **changes** with the treatment | Only with no unmeasured confounding | A randomised trial, or causal methods |

A **randomised trial** chooses the treatment at random, for example with a coin toss. Then no patient fact can change the treatment choice, so there is no confounding. **Causal methods** try to copy a trial with recorded data. Two examples are target-trial emulation and propensity methods. They still need the no-unmeasured-confounding assumption.

DCA compares one action with no action. "Treatment A instead of treatment B" is not one action that a risk cut-off can start. So the DCA in this notebook cannot check treatment advice. DCA can check a model of treatment benefit, but only if that model comes from a causal method.

## Part 3 · The same warning signs in the CIBMTR data

Now we use the competition data. Here we have no answer key. Each patient has an outcome only for the treatment they got.

The table below compares the MAC and RIC groups. For each group, it shows:
- the average age at transplant (`mean_age_at_hct`),
- the average comorbidity score (`mean_comorbidity_score`). A **comorbidity** is another illness that the patient has. A higher score means more illness.
- the average Karnofsky score (`mean_karnofsky_score`). This score measures how well the patient can do daily activities, from 0 to 100. A higher score means a fitter patient.
- the number of patients (`n`) and the percent with an event (`event_pct`).

In [ ]:
example = project.treatment_example
train = load_train(project)
teaching.arm_case_mix(train, example.column, example.arms, example.case_mix, project.columns.event).round(2)

RIC patients have more events. They are also older, have more other illnesses and are less fit (lower Karnofsky score). This is the same pattern as Part 1. So the naive comparison has confounding.

A model can correct for age, comorbidity and Karnofsky score, because they are recorded. But doctors also choose by disease status, by details of the patient's fitness, and by what the patient wants. Not all of this is in the data. So the model cannot correct for all of it.

Also, this data is **synthetic**. A program (SurvivalGAN) made it from real records. A treatment effect in this data shows what the program learned. It is not a medical fact.

## What this means for the CIBMTR summary

1. We check our models as **risk** models:
   - **Discrimination**: how well the model puts patients in order of risk.
   - **Calibration**: how close the predicted risks are to the actual event rates, in each race group.
   - **DCA**: if the model helps act-or-not decisions.
2. We do **not** say that any model shows which conditioning, donor or graft is better.
3. The limits section of the summary must say this. A treatment comparison needs a causal method.